# 3. The baseline control law

Two questions for the inherited current schedule: does it lift the pod on its
own, and can the feedback loops on top of it cover the gap where it falls short.

The baseline schedule (`track/control.py::assign_currents_for_pod`, "Eq. 18" in
the writeup) does not levitate the pod under any preset tested. The lift it
produces is smaller than the pod's weight by more than a factor of a thousand.

The first experiment is a 300 ms lateral step. The pod starts 6.40 mm from its
target and the lateral PD loop is asked to bring it back. The same run is
repeated in three configurations. Track, pod, the 6.40 mm offset, lateral
`kp = 180` and `force_limit = 3.2 N` are held fixed; only the vertical loop and
the lateral damping gain change:

| | vertical loop | lateral `kd` | height |
|---|---|---|---|
| **A** | open (`Fz = 0`) | 5 | free-falls the full 40 mm, ending about 390 mm below the track |
| **B** | engaged, re-levitated every 5 ms | 5 (shipped value) | held at 40 mm |
| **C** | engaged, re-levitated every 5 ms | 20 | held at 40 mm |

Configuration A is the experiment as originally shipped. With nothing holding the
pod up it falls throughout the run, so the lateral error at the end is mostly the
pod coasting out of the usable field rather than station-keeping; section 2.3.3
of the writeup warns about this confound. B and C hold the height fixed, so the
lateral figure at 300 ms reflects the lateral loop alone.

All three runs share the same early transient: between about 60 and 90 ms the
error drops from 6.40 mm to well under a millimetre. That transient measures
local lateral authority. Whether the loop then holds the pod in place is a
separate question, and only B and C can answer it.

In [1]:
from nblib.bootstrap import REPO_ROOT

import numpy as np
import matplotlib.pyplot as plt

from config import SimConfig
from config import G

from data_io.presets import load_track_preset, load_pod_preset
from geometry.sampling import generate_pod_base_points
from physics.dipole import dipole_moment
from physics.field_query import field_gradient_at_point
from physics.linearise import build_total_position_stiffness
from track.control import assign_currents_for_pod

from simulation.calibration import (
    calibrate_levitation, local_current_scale,
    calibrate_for_target, make_live_scale_fn,
)
from simulation.dynamics import DynamicsState, DynamicsStateSO3
from simulation.closed_loop import run_closed_loop_dynamics
from simulation.closed_loop_so3 import run_closed_loop_dynamics_so3
from control.allocation import CurrentAllocator
from control.xy_controller import LateralTarget, LateralPDController
from control.z_controller import ZTarget, ZPDController
from control.combined_controller import CombinedController

FIG_DIR = REPO_ROOT / "docs" / "results_figures"


def _total_gradient(magnets, point):
    """Add up every magnet's field-gradient contribution at `point` and return
    the 3x3 total."""
    grad = np.zeros((3, 3))
    for magnet in magnets:
        grad += field_gradient_at_point(magnet, point)
    return grad

In [ ]:
# 300 ms lateral run. The setup below is shared by configurations A, B and C.
TRACK_NAME, POD_NAME = "Classic Track", "Two Stacks"
X0, Y0, Z0 = 0.0, 0.001, 0.040
TARGET_X, TARGET_Y = 0.005, -0.003          # hypot -> 6.40 mm initial offset
KP, FORCE_LIMIT = 180.0, 3.2               # lateral gains, shared by all three runs
ZKP, ZKD, ZFORCE_LIMIT = 5000.0, 20.0, 1.0  # vertical loop, the shipped ClosedLoopFull set
T_SPAN, CONTROL_DT, MAX_STEP = (0.0, 0.3), 0.005, 0.001

CFG = SimConfig()


def _load():
    track, _ = load_track_preset(TRACK_NAME)
    pod = load_pod_preset(POD_NAME)
    return track, pod, generate_pod_base_points(pod)


def _summary(sol):
    """Pull (time in ms, lateral error in mm, height in mm) out of a solved run.
    Rows 0-2 of `sol.y` are x, y, z for both the Euler and the SO(3) result
    objects."""
    t_ms = sol.t * 1000.0
    dist_mm = np.hypot(TARGET_X - sol.y[0], TARGET_Y - sol.y[1]) * 1000.0
    return t_ms, dist_mm, sol.y[2] * 1000.0


def run_freefall():
    """Configuration A: the experiment as shipped, lateral PD only, the
    uncalibrated baseline, and the Euler runner. Nothing holds the pod up, so it
    free-falls (Fz = 0)."""
    track, pod, pod_pts = _load()
    ctrl = LateralPDController(target=LateralTarget(x=TARGET_X, y=TARGET_Y),
                              kp=KP, kd=5.0, force_limit=FORCE_LIMIT)
    allocator = CurrentAllocator(current_limit=0.5, fz_weight=100.0)
    return run_closed_loop_dynamics(
        pod, pod_pts, track, ctrl, allocator,
        DynamicsState(x0=X0, y0=Y0, z0=Z0), CFG,
        t_span=T_SPAN, control_dt=CONTROL_DT, max_step=MAX_STEP,
    )


def run_vertical_engaged(lateral_kd):
    """Configurations B and C: lateral PD plus a vertical PD loop. The baseline
    is re-levitated at the live pose on every control step, which pins z at Z0.
    This needs the SO(3) runner because `current_scale` is passed as a callable."""
    track, pod, pod_pts = _load()
    target_pod_xy = float(np.hypot(TARGET_X, TARGET_Y))
    cal, current_limit = calibrate_for_target(track, pod, pod_pts, CFG, target_pod_xy)

    controller = CombinedController([
        LateralPDController(target=LateralTarget(x=TARGET_X, y=TARGET_Y),
                            kp=KP, kd=lateral_kd, force_limit=FORCE_LIMIT),
        ZPDController(target=ZTarget(z=Z0), kp=ZKP, kd=ZKD, force_limit=ZFORCE_LIMIT),
    ])
    allocator = CurrentAllocator(current_limit=current_limit, fz_weight=100.0,
                                 solver="bounded")
    scale_fn = make_live_scale_fn(track, pod, CFG, allocator, cal.current_scale)

    return run_closed_loop_dynamics_so3(
        pod, pod_pts, track, controller, allocator,
        DynamicsStateSO3.from_euler(x0=X0, y0=Y0, z0=Z0,
                                    roll0=0.0, pitch0=0.0, yaw0=0.0),
        CFG, t_span=T_SPAN, control_dt=CONTROL_DT, max_step=MAX_STEP,
        current_scale=scale_fn, angular_damping=0.01,
    )

In [3]:
sol_A = run_freefall()
sol_B = run_vertical_engaged(lateral_kd=5.0)
sol_C = run_vertical_engaged(lateral_kd=20.0)

for name, sol in [("A free fall,  kd=5 ", sol_A),
                  ("B height held, kd=5 ", sol_B),
                  ("C height held, kd=20", sol_C)]:
    t, d, z = _summary(sol)
    i300 = int(np.argmin(np.abs(t - 300.0)))
    print(f"{name}: err {d[0]:.2f} -> min {d.min():.2f}mm @ {t[np.argmin(d)]:.0f}ms "
          f"-> {d[i300]:.2f}mm @ 300ms   z [{z.min():.1f} .. {z.max():.1f}]mm")

A free fall,  kd=5 : err 6.40 -> min 0.17mm @ 92ms -> 2.41mm @ 300ms   z [-389.7 .. 40.0]mm
B height held, kd=5 : err 6.40 -> min 0.19mm @ 64ms -> 5.16mm @ 300ms   z [40.0 .. 40.0]mm
C height held, kd=20: err 6.40 -> min 0.62mm @ 217ms -> 1.63mm @ 300ms   z [40.0 .. 40.0]mm


In [ ]:
tA, dA, zA = _summary(sol_A)
tB, dB, zB = _summary(sol_B)
tC, dC, zC = _summary(sol_C)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7.4), sharex=True,
                               gridspec_kw={"height_ratios": [1, 1.25]})

# Top: lateral error for the three runs.
ax1.plot(tA, dA, color="grey", lw=2.0, label="A  free fall, kd = 5")
ax1.plot(tB, dB, color="orange", lw=2.3, label="B  height held, kd = 5")
ax1.plot(tC, dC, color="green", lw=2.6, label="C  height held, kd = 20")
ax1.set_ylabel("lateral error (mm)")
ax1.legend(fontsize=9, frameon=False)
ax1.grid(alpha=0.25)
for s in ("top", "right"):
    ax1.spines[s].set_visible(False)

# Bottom: height. A free-falls; B and C are pinned at 40 mm.
ax2.axhline(0.0, color="blue", lw=1.4, zorder=2)
ax2.text(tA[0], 4, "track plane (z = 0)", fontsize=9, color="blue", va="bottom")
ax2.plot(tA, zA, color="grey", lw=2.0, label="A  free fall")
ax2.plot(tB, zB, color="green", lw=2.6, label="B, C  height held (z = 40 mm)")
ax2.set_ylabel("z (mm)")
ax2.set_xlabel("t (ms)")
ax2.legend(fontsize=9, frameon=False, loc="lower left")
ax2.grid(alpha=0.25)
for s in ("top", "right"):
    ax2.spines[s].set_visible(False)

fig.suptitle("300 ms lateral run: free-fall reference vs vertical loop engaged",
             fontsize=13, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIG_DIR / "fig2c_lateral_vertical_rerun.png", dpi=150)
plt.show()

Reading the three traces:

**A, free fall.** The error dips to 0.17 mm at about 92 ms, but by then the pod
is roughly 390 mm below the track and still dropping. The 2.41 mm reading at
300 ms is an artefact of that fall, not station-keeping.

**B, height held, shipped `kd = 5`.** The 60-90 ms transient drives the error
down to 0.19 mm, so the actuators produce the corrective force the controller
asks for; the allocator realises it to within about 1%. But `kd = 5` is
under-damped against the +42 N/m anti-restoring stiffness of the baseline
schedule, so the pod overshoots, climbs back to about 5 mm by 300 ms, and
diverges if the window is extended.

**C, height held, `kd = 20`.** With the height held and enough damping, the error
decays monotonically to about 1.6 mm at 300 ms.

So the loop has local lateral authority at the operating point, but the shipped
gains do not hold station once the free fall is taken out. The PD gains have not
been calibrated against validated rig data (see the Limitations section), and no
claim of closed-loop stability is made here.

## 3a. The lift curve

`calibrate_levitation` inspects the schedule and returns three things: the height
`z_ref` where the lift peaks, a nearby point `z_stable` just past the peak where
the force is locally restoring, and a `current_scale` factor that multiplies the
raw schedule so the peak lift comes out at `margin` times the pod's weight.

That scale factor is large. Before scaling, the peak lift is hundreds of times
smaller than the weight; the factor is printed below. The open-loop `z(t)` traces
elsewhere in the submission are flat because the height was pinned by
construction, not because the schedule can hold the pod there on its own.

In [ ]:
cfg = SimConfig()
track_k, _ = load_track_preset("Classic Track")
pod_k = load_pod_preset("Two Stacks")
pod_pts_k = generate_pod_base_points(pod_k)
m_body_k = dipole_moment(pod_k, pod_pts_k)
weight_k = pod_k.TotalMass * G

cal_k = calibrate_levitation(track_k, pod_k, pod_pts_k, cfg, pod_xy=0.0)
assign_currents_for_pod(track_k, pod_xy=0.0, pod_total_mass=pod_k.TotalMass,
                        config=cfg, current_scale=cal_k.current_scale)

# Lift force along the centreline over a range of heights.
zs_k = np.linspace(0.6 * cal_k.z_ref, 1.8 * cal_k.z_stable, 200)
Fz_k = np.array([
    (m_body_k @ _total_gradient(track_k.Magnets, np.array([0.0, 0.0, z])))[2]
    for z in zs_k
])

short_by = weight_k / cal_k.Fz_at_peak_unscaled
print(f"z_ref = {cal_k.z_ref*1000:.1f} mm, z_stable = {cal_k.z_stable*1000:.1f} mm; "
      f"unscaled peak lift is {short_by:.0f}x short of weight ({weight_k:.3f} N)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(zs_k * 1000, Fz_k, color="blue", lw=2.2, label="scaled schedule $F_z(z)$")
ax.axhline(weight_k, color="red", lw=1.6, ls="--", label=f"pod weight ({weight_k:.2f} N)")
ax.axvline(cal_k.z_ref * 1000, color="grey", lw=1.4, ls=":", label="$z_{ref}$ (peak, half-stable)")
ax.axvline(cal_k.z_stable * 1000, color="green", lw=1.4, ls=":", label="$z_{stable}$ (locally restoring)")

ax.set_xlabel("z (mm)")
ax.set_ylabel("$F_z$ (N)")
ax.set_title(f"Baseline lift curve $F_z(z)$ "
             f"(current_scale = {cal_k.current_scale:.1f}x)")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_levitation_fz_curve.png", dpi=150)
plt.show()

The peak of the lift curve is only half-stable: it restores against an upward
displacement but not a downward one. `z_stable` sits further out, on the
descending branch where `dFz/dz < 0`, and that is where the vertical loop is
trimmed to operate.

## 3b. Stiffness and trim away from the centreline

`build_total_position_stiffness` re-solves the baseline schedule at each
perturbed position rather than reusing the centre solution, so the `dFx/dx` it
returns is the lateral stiffness a PD loop has to fight. That stiffness is
positive (anti-restoring) at the centre and climbs steeply as the pod moves
off-axis.

The vertical trim is similar. Along the twin-stack axis, the `local_current_scale`
needed to keep `Fz == weight` swings several-fold over a few millimetres. Both
curves say the same thing: the baseline trim is only valid dead centre.

In [ ]:
track_b, _ = load_track_preset("Classic Track")
pod_b = load_pod_preset("Two Stacks")
pod_pts_b = generate_pod_base_points(pod_b)
m_world_b = dipole_moment(pod_b, pod_pts_b)          # level attitude

cal_b = calibrate_levitation(track_b, pod_b, pod_pts_b, cfg, pod_xy=0.0)
z0_b = cal_b.z_stable

xs_b = np.linspace(1e-4, 0.02, 25)
k_xx = []
trim = []
for x in xs_b:
    pos = np.array([x, 0.0, z0_b])
    J = build_total_position_stiffness(track_b, pos, m_world_b, pod_b.TotalMass, cfg,
                                       current_scale=cal_b.current_scale)
    k_xx.append(J[0, 0])
    try:
        trim.append(local_current_scale(track_b, pos, m_world_b, pod_b.TotalMass, cfg))
    except ValueError:
        trim.append(np.nan)

k_xx = np.array(k_xx)
trim = np.array(trim)

sign = "anti-restoring (destabilising)" if k_xx[0] > 0 else "restoring"
print(f"dFx/dx near centre = {k_xx[0]:+.1f} N/m, {sign} "
      f"(the writeup quotes +38.8 N/m); at x = 20 mm it is {k_xx[-1]:+.1f} N/m")

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))

# Left axis (red): lateral stiffness dFx/dx.
ax1.axhline(0.0, color="grey", lw=1.0)
ax1.plot(xs_b * 1000, k_xx, color="red", lw=2.2, marker="o", markersize=3)
ax1.set_xlabel("lateral offset x (mm)")
ax1.set_ylabel("$dF_x/dx$ (N/m)", color="red")
ax1.tick_params(axis="y", labelcolor="red")
ax1.set_title("Lateral stiffness and vertical trim vs offset")
ax1.grid(alpha=0.3)

# Right axis (blue): vertical trim scale needed to keep Fz == weight.
ax2 = ax1.twinx()
ax2.semilogy(xs_b * 1000, trim, color="blue", lw=2.2, marker="s", markersize=3)
ax2.set_ylabel("vertical trim scale (x, log)", color="blue")
ax2.tick_params(axis="y", labelcolor="blue")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_baseline_offcentre_stiffness.png", dpi=150)
plt.show()

Near the centre `dFx/dx` is about +42 N/m. The sign is what matters: it is
positive, so the schedule's own lateral force pushes the pod further off-centre
as soon as it is displaced. Supplying the restoring force the baseline lacks is
the job of the lateral loop.